# 🛒 Caso Olist — Diagnóstico de Dados
## Dois problemas reais. Uma solução em Python.

---

> *"Vocês são a nova equipe de Ciência de Dados da Olist.  
> Regra de ouro: nós não programamos por programar — usamos dados para resolver problemas reais."*

---

## 📬 Os chamados que chegaram hoje

### 📣 Chamado 1 — Diretora de Marketing
> *"A Black Friday está chegando! Quero mandar cupom de 30% para clientes recorrentes.  
> Mas o dashboard mostra **99.441 pedidos** e **99.441 clientes**. Conclusão: **ninguém volta para comprar com a gente!**  
> Nossa taxa de retenção é ZERO!"*

### 🚚 Chamado 2 — Diretor de Logística
> *"Nosso painel de tempo médio de entrega está **quebrando e dando erros absurdos**. Precisamos corrigir antes da reunião de sexta."*

---

## 🎯 Plano de ataque

| # | Problema | Suspeita | Ferramenta |
|---|---|---|---|
| 1 | Marketing vê retenção zero | Está olhando para o dado errado (`customer_id`) | `nunique()`, `value_counts()` |
| 2 | Logística com erros matemáticos | Datas como texto + valores nulos | `pd.to_datetime()`, `dropna()` |
| 3 | Criar lista VIP para campanha | Cruzar 3 tabelas com segurança | `pd.merge()` (LEFT JOIN) |

---

## ⚙️ Setup — Importações e carregamento dos dados

In [ ]:
# ============================================================
# CÉLULA 1 — Importações e Camada Bronze
# ============================================================
import pandas as pd

# Ajuste o caminho conforme seu ambiente:
#   Google Colab  → DATA_PATH = '/content/'
#   Google Drive  → DATA_PATH = '/content/drive/MyDrive/olist/'
#   Local         → DATA_PATH = './data/'
DATA_PATH = './data/'

# Carregamos SEM nenhuma transformação — dados brutos (Camada Bronze)
df_orders    = pd.read_csv(DATA_PATH + 'olist_orders_dataset.csv')
df_customers = pd.read_csv(DATA_PATH + 'olist_customers_dataset.csv')
df_items     = pd.read_csv(DATA_PATH + 'olist_order_items_dataset.csv')
df_payments  = pd.read_csv(DATA_PATH + 'olist_order_payments_dataset.csv')

print('✅ Dados carregados!')
print(f'   orders:    {df_orders.shape[0]:,} linhas')
print(f'   customers: {df_customers.shape[0]:,} linhas')
print(f'   items:     {df_items.shape[0]:,} linhas')
print(f'   payments:  {df_payments.shape[0]:,} linhas')

---
# 🔍 Problema 1 — O Paradoxo do Cliente
### *"A taxa de retenção é zero?"*

A Diretora comparou número de **pedidos** com número de **clientes** e concluiu que ninguém compra duas vezes.
Vamos reproduzir o raciocínio dela para entender onde está o erro.

### A raiz do problema: duas colunas que parecem iguais, mas não são

| Coluna | O que representa | Analogia |
|---|---|---|
| `customer_id` | **Chave do pedido** — gerada a cada nova compra | Número de protocolo de atendimento |
| `customer_unique_id` | **Identidade real** do cliente (equivale ao CPF) | O próprio CPF |

> **Exemplo:** se Maria comprou 3 vezes, ela tem **3 `customer_id`** diferentes, mas apenas **1 `customer_unique_id`**.

In [ ]:
# ============================================================
# CÉLULA 2 — Reproduzindo o raciocínio (errado) do Marketing
# ============================================================
# A Diretora comparou linhas da tabela customers com linhas de orders.
# Vamos ver o que ela estava enxergando:

total_pedidos      = len(df_orders)
total_linhas_cust  = len(df_customers)  # linhas brutas da tabela

print('📊 O que a Diretora de Marketing viu no dashboard:')
print(f'   Total de pedidos registrados : {total_pedidos:,}')
print(f'   Total de linhas em customers : {total_linhas_cust:,}')
print()
print('❌ Conclusão ERRADA dela: "1 pedido por cliente = retenção zero!"')
print()
print('⚠️  Mas por que os dois números são EXATAMENTE iguais?')
print('   Isso não é coincidência — é a pista que revela o problema.')

In [ ]:
# ============================================================
# CÉLULA 3 — Investigando com nunique()
# ============================================================
# nunique() conta o número de valores DISTINTOS em uma coluna.
# Equivalente ao COUNT(DISTINCT coluna) do SQL.
# Aqui vemos a diferença REAL entre as duas colunas de identificação.

n_customer_id     = df_customers['customer_id'].nunique()
n_unique_customer = df_customers['customer_unique_id'].nunique()

print('=' * 58)
print('         🧩 O PARADOXO DO CLIENTE — REVELADO')
print('=' * 58)
print(f'  Linhas na tabela customers          : {len(df_customers):>10,}')
print(f'  Valores únicos de customer_id       : {n_customer_id:>10,}')
print(f'  Valores únicos de customer_unique_id: {n_unique_customer:>10,}')
print('=' * 58)

diferenca = n_customer_id - n_unique_customer
print(f'\n  ✅ Diferença: {diferenca:,} registros a mais que clientes reais')
print(f'  → Esses {diferenca:,} são de clientes que compraram MAIS DE UMA VEZ!')

In [ ]:
# ============================================================
# CÉLULA 4 — Quantos clientes são realmente recorrentes?
# ============================================================
# value_counts() conta a frequência de cada valor único.
# Aplicamos sobre customer_unique_id para ver quantas vezes
# cada cliente aparece. Mais de 1 aparição = cliente recorrente.

pedidos_por_cliente = df_customers['customer_unique_id'].value_counts()

# Filtrando clientes com mais de 1 pedido
clientes_recorrentes = pedidos_por_cliente[pedidos_por_cliente > 1]
pct_recorrentes = len(clientes_recorrentes) / n_unique_customer * 100

print('🎉 O que REALMENTE está acontecendo:')
print(f'   Clientes únicos reais           : {n_unique_customer:,}')
print(f'   Clientes com mais de 1 compra   : {len(clientes_recorrentes):,} ({pct_recorrentes:.1f}%)')
print()
print('📌 Top 10 clientes mais recorrentes:')
print(clientes_recorrentes.head(10).to_frame('nº de pedidos').to_string())
print()
print('⭐ Um cliente chegou a comprar', pedidos_por_cliente.max(), 'vezes!')

In [ ]:
# ============================================================
# CÉLULA 5 — Distribuição de frequência de compras
# ============================================================
# Aplicamos value_counts() sobre o resultado de outro value_counts():
# → Primeiro: quantos pedidos cada cliente fez
# → Depois: quantos clientes fizeram exatamente N pedidos

dist_compras = pedidos_por_cliente.value_counts().sort_index()

print('📊 Distribuição: quantos clientes fizeram X compras?')
print()
print(f'{"Nº de compras":<18} {"Nº de clientes":>15} {"% do total":>12}')
print('-' * 48)
for n_compras, n_clientes in dist_compras.items():
    pct = n_clientes / n_unique_customer * 100
    barra = '█' * int(pct / 1.5)
    print(f'{n_compras:<18} {n_clientes:>15,} {pct:>11.1f}%  {barra}')

### 💼 Impacto no Negócio: por que isso importa?

Usar o dado errado não é só um erro técnico — afeta decisões estratégicas de alto nível:

**CAC — Custo de Aquisição de Clientes**
```
CAC = Total gasto em Marketing / Nº de novos clientes

❌ Com customer_id (errado): CAC = R$ 500.000 / 99.441 = R$ 5,03
✅ Com customer_unique_id  : CAC = R$ 500.000 / 96.096 = R$ 5,20  ← mais caro do que parecia
```

**LTV — Life Time Value**
```
LTV = Soma de toda a receita gerada por um cliente ao longo do tempo

❌ Impossível com customer_id — cada compra parece ser de um novo cliente!
✅ Com customer_unique_id: somamos todos os pedidos do mesmo CPF.
```

> **✅ Conclusão para o Marketing:** A Diretora estava **errada**. Temos **2.997 clientes recorrentes** prontos para receber o cupom. A campanha está salva! 🎉

---
# 🚚 Problema 2 — Os Erros do Diretor de Logística
### *"Os cálculos de tempo de entrega estão quebrando"*

Antes de investigar os nulos, vamos entender **por que o código quebra** — o problema começa nos tipos de dados.

In [ ]:
# ============================================================
# CÉLULA 6 — Reproduzindo o erro: o problema começa nos tipos
# ============================================================
# df.info() mostra o tipo (dtype) de cada coluna.
# Procure as colunas de data: elas aparecem como 'object' (texto)!

print('🔍 Tipos de dados da tabela ORDERS:')
df_orders.info()
print()
print('⚠️  Colunas de data aparecem como dtype object = texto!')
print('   Python não consegue subtrair textos — daí o erro.')

In [ ]:
# ============================================================
# CÉLULA 7 — Demonstrando o erro na prática
# ============================================================
# Veja o que acontece ao tentar calcular tempo de entrega
# com as colunas ainda como texto (object):

try:
    resultado = (df_orders['order_delivered_customer_date']
                 - df_orders['order_purchase_timestamp'])
    print('Funcionou:', resultado)
except TypeError as e:
    print('❌ ERRO que o Diretor está vendo:')
    print(f'   TypeError: {e}')
    print()
    print('   Python não sabe subtrair strings.')
    print('   Precisamos converter para datetime antes.')

In [ ]:
# ============================================================
# CÉLULA 8 — Solução: pd.to_datetime()
# ============================================================
# pd.to_datetime() converte uma coluna de texto para datetime64.
# Com o tipo correto, o Python consegue fazer aritmética de datas.
#
# errors='coerce' → valores inválidos viram NaT (Not a Time)
# ao invés de lançar um erro e travar a execução.

df_orders_clean = df_orders.copy()  # boa prática: nunca modificar o original

colunas_data = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in colunas_data:
    df_orders_clean[col] = pd.to_datetime(df_orders_clean[col], errors='coerce')

print('✅ Colunas convertidas:')
print(df_orders_clean[colunas_data].dtypes)
print()
print('Agora o dtype é datetime64[ns] — pronto para aritmética!')

### 🕳️ O segundo problema: Valores Nulos (NaN)

Mesmo com os tipos corretos, ainda há pedidos **sem data de entrega**. Por quê?

| Motivo | Status típico |
|---|---|
| Pedido ainda em trânsito | `shipped` |
| Pedido cancelado antes de sair | `canceled` |
| Erro no sistema da transportadora | `delivered` com data nula |
| Produto indisponível | `unavailable` |

> ⚠️ Esta não é uma **decisão de código** — é uma **decisão de negócio**:  
> dropar ou imputar depende de **qual pergunta você está respondendo**.

In [ ]:
# ============================================================
# CÉLULA 9 — Mapeando os valores nulos em orders
# ============================================================
# isnull() retorna True para cada célula vazia (NaN / NaT).
# .sum() soma os True por coluna → total de nulos por coluna.
# Dividimos pelo total de linhas para obter a porcentagem.

nulos = df_orders_clean.isnull().sum()
pct   = (nulos / len(df_orders_clean) * 100).round(2)

df_nulos = pd.DataFrame({'Nulos': nulos, '% do Total': pct})

print('🕵️ Valores nulos na tabela ORDERS:')
print(df_nulos[df_nulos['Nulos'] > 0].to_string())
print()
n_nulos = nulos['order_delivered_customer_date']
pct_n   = pct['order_delivered_customer_date']
print(f'📌 order_delivered_customer_date: {n_nulos:,} nulos ({pct_n}% dos pedidos)')

In [ ]:
# ============================================================
# CÉLULA 10 — Investigando quem são os pedidos sem data de entrega
# ============================================================
# Antes de decidir o que fazer com os nulos, entendemos
# o STATUS desses pedidos. Se são cancelados, faz todo sentido
# não ter data de entrega — e devem sair do cálculo logístico.

pedidos_sem_entrega = df_orders_clean[
    df_orders_clean['order_delivered_customer_date'].isnull()
]

print(f'📦 Os {len(pedidos_sem_entrega):,} pedidos SEM data de entrega, por status:')
print()
print(pedidos_sem_entrega['order_status'].value_counts().to_string())
print()
print('💡 Conclusão: são pedidos cancelados, em trânsito ou com problemas.')
print('   Correto não ter data de entrega. Para lead time, filtramos eles.')

In [ ]:
# ============================================================
# CÉLULA 11 — Abordagem 1: dropna() — O Extermínio
# ============================================================
# dropna() remove linhas que contêm NaN na coluna especificada.
#
# QUANDO USAR: quando a linha sem dado não faz sentido para a análise.
# CASO LOGÍSTICO: calcular tempo de entrega sem data de entrega
# é impossível → dropna é a escolha correta.
#
# subset=['coluna'] → remove só onde ESSA coluna específica é nula

df_entregues = df_orders_clean.dropna(
    subset=['order_delivered_customer_date']
).copy()

print('🗑️  Abordagem 1 — dropna() (Extermínio):')
print(f'   Linhas ANTES : {len(df_orders_clean):>10,}')
print(f'   Linhas DEPOIS: {len(df_entregues):>10,}')
print(f'   Removidas    : {len(df_orders_clean) - len(df_entregues):>10,}')
print()
print('✅ Correto para logística — pedidos não entregues fora do cálculo.')

In [ ]:
# ============================================================
# CÉLULA 12 — Abordagem 2: fillna() — A Imputação
# ============================================================
# fillna() preenche os NaN com um valor definido.
#
# QUANDO USAR: quando a linha tem dados úteis em outras colunas
# e NÃO pode ser descartada.
#
# EXEMPLO CRÍTICO: calcular receita total da empresa.
# Um pedido R$ 5.000 em trânsito tem payment_value preenchido.
# Usar dropna na data de entrega apagaria essa receita do painel!

df_financeiro = df_orders_clean.copy()

# Criamos um flag para não perder a linha, apenas sinalizá-la
df_financeiro['status_entrega'] = df_financeiro['order_delivered_customer_date'].apply(
    lambda x: 'Entregue' if pd.notna(x) else 'Pendente'
)

print('💰 Abordagem 2 — fillna() / flag (Imputação):')
print(df_financeiro['status_entrega'].value_counts().to_string())
print()
print('⚠️  Com dropna na data de entrega, perderíamos receita de pedidos')
print('   Pendentes no cálculo financeiro — erro silencioso e perigoso!')

In [ ]:
# ============================================================
# CÉLULA 13 — Calculando o tempo de entrega (corretamente!)
# ============================================================
# Com os tipos corretos E os nulos removidos, podemos subtrair datas.
# O resultado é um timedelta → .dt.days extrai o número inteiro de dias.

# Tempo total: compra → recebimento pelo cliente
df_entregues['tempo_entrega_dias'] = (
    df_entregues['order_delivered_customer_date']
    - df_entregues['order_purchase_timestamp']
).dt.days

# Tempo até despacho: compra → entrega ao carrier
df_entregues['tempo_despacho_dias'] = (
    df_entregues['order_delivered_carrier_date']
    - df_entregues['order_purchase_timestamp']
).dt.days

print('✅ PROBLEMA DO DIRETOR DE LOGÍSTICA — RESOLVIDO!')
print()
print('📦 Tempo de Entrega Total (compra → cliente):')
print(f'   Média   : {df_entregues["tempo_entrega_dias"].mean():.1f} dias')
print(f'   Mediana : {df_entregues["tempo_entrega_dias"].median():.0f} dias')
print(f'   Mínimo  : {df_entregues["tempo_entrega_dias"].min()} dias')
print(f'   Máximo  : {df_entregues["tempo_entrega_dias"].max()} dias')
print()
print('🚛 Tempo até Despacho (compra → carrier):')
print(f'   Média   : {df_entregues["tempo_despacho_dias"].mean():.1f} dias')
print(f'   Mediana : {df_entregues["tempo_despacho_dias"].median():.0f} dias')

---
# 🔗 Problema 3 — A Lista VIP para a Black Friday
### *Cruzando tabelas com `pd.merge()`*

Temos os clientes recorrentes identificados. Agora precisamos saber **quanto cada um gastou** para montar a lista da campanha. Isso exige cruzar 3 tabelas.

### A lógica dos JOINs

```
customers ──── orders ──── payments
    ↕               ↕
customer_id      order_id
```

| Tipo | O que faz |
|---|---|
| `INNER JOIN` | Só o que existe nos **dois** lados — descarta o resto |
| `LEFT JOIN` | Tudo da esquerda + o que bater à direita — a tabela guia nunca perde linhas |
| `OUTER JOIN` | Tudo dos dois lados, com NaN onde não casar |

In [ ]:
# ============================================================
# CÉLULA 14 — ⚠️ A Armadilha do Produto Cartesiano
# ============================================================
# ATENÇÃO: esta célula demonstra um ERRO INTENCIONAL.
# Um pedido pode ter VÁRIOS itens E VÁRIAS formas de pagamento.
# Fazer merge direto entre essas duas tabelas multiplica as linhas!

# ❌ ERRADO: merge direto sem pré-agregar
df_errado = pd.merge(df_items, df_payments, on='order_id', how='inner')

receita_errada  = df_errado['payment_value'].sum()
receita_correta = df_payments['payment_value'].sum()

print('🚨 ARMADILHA DO PRODUTO CARTESIANO')
print('=' * 52)
print(f'  Linhas em order_items    : {len(df_items):>10,}')
print(f'  Linhas em order_payments : {len(df_payments):>10,}')
print(f'  Linhas após merge ERRADO : {len(df_errado):>10,}  ← se multiplicou!')
print('=' * 52)
print(f'  Receita CORRETA  : R$ {receita_correta:>15,.2f}')
print(f'  Receita ERRADA   : R$ {receita_errada:>15,.2f}  ← inflada!')
print(f'  Fator de inflação: {receita_errada/receita_correta:.2f}x')
print()
print('💡 Regra: SEMPRE verifique df.shape antes e depois de um merge!')
print('   Pré-agrege a tabela mais granular antes de cruzar.')

In [ ]:
# ============================================================
# CÉLULA 15 — ✅ Merge Correto: pré-agregar antes de cruzar
# ============================================================
# Solução: colapsamos payments para 1 linha por order_id ANTES do merge.
# Assim garantimos a relação 1-para-1 com a tabela orders.
#
# groupby('order_id'): agrupa todas as linhas com o mesmo order_id
# agg(): define como resumir cada coluna dentro do grupo
# reset_index(): transforma o índice do groupby em coluna normal

payments_por_pedido = df_payments.groupby('order_id').agg(
    total_pago     = ('payment_value', 'sum'),      # soma todas as parcelas
    n_parcelas     = ('payment_installments', 'max'),
    tipo_pagamento = ('payment_type', 'first')      # tipo da 1ª transação
).reset_index()

print('✅ payments pré-agregado: 1 linha por pedido')
print(f'   Antes: {len(df_payments):,} linhas → Depois: {len(payments_por_pedido):,} linhas')
print()
print(payments_por_pedido.head(3).to_string())

In [ ]:
# ============================================================
# CÉLULA 16 — Construindo a cadeia de JOINs
# ============================================================
# Agora montamos a visão completa em 2 merges encadeados.
#
# pd.merge(left, right, on='chave', how='tipo')
#   left  → tabela base (guia)
#   right → tabela a adicionar
#   on    → coluna em comum (a chave de ligação)
#   how   → 'left', 'inner', 'outer' ou 'right'

# MERGE 1: orders + customers (LEFT JOIN pelo customer_id)
# → Preservamos todos os pedidos e adicionamos dados do cliente
df_step1 = pd.merge(
    df_orders[['order_id', 'customer_id']],
    df_customers[['customer_id', 'customer_unique_id', 'customer_city', 'customer_state']],
    on='customer_id',
    how='left'
)
print(f'Merge 1 (orders + customers): {len(df_step1):,} linhas ✅')

# MERGE 2: resultado + payments pré-agregado (LEFT JOIN pelo order_id)
# → Adicionamos o valor total pago por pedido
df_full = pd.merge(
    df_step1,
    payments_por_pedido[['order_id', 'total_pago', 'tipo_pagamento']],
    on='order_id',
    how='left'
)
print(f'Merge 2 (+ payments):         {len(df_full):,} linhas ✅')
print()
print('🔎 Amostra do resultado final:')
df_full.head(3)

In [ ]:
# ============================================================
# CÉLULA 17 — 🏆 Gerando a Lista VIP para a Black Friday
# ============================================================
# Agrupamos por customer_unique_id para contar pedidos e somar receita.
# .query('total_pedidos > 1') → filtra apenas clientes recorrentes.
# .sort_values('total_gasto', ascending=False) → maiores gastadores primeiro.

lista_vip = (
    df_full
    .groupby('customer_unique_id')
    .agg(
        total_pedidos = ('order_id', 'count'),
        total_gasto   = ('total_pago', 'sum'),
        estado        = ('customer_state', 'first')
    )
    .reset_index()
    .query('total_pedidos > 1')           # apenas recorrentes
    .sort_values('total_gasto', ascending=False)
    .reset_index(drop=True)
)

lista_vip['total_gasto'] = lista_vip['total_gasto'].round(2)

print('🏆 LISTA VIP — Clientes Recorrentes para a Black Friday')
print('=' * 60)
print(f'  Total de clientes elegíveis : {len(lista_vip):,}')
print(f'  Gasto médio por cliente VIP : R$ {lista_vip["total_gasto"].mean():,.2f}')
print(f'  Maior gasto (top cliente)   : R$ {lista_vip["total_gasto"].max():,.2f}')
print('=' * 60)
print()
print('🥇 Top 10 clientes VIP por total gasto:')
lista_vip.head(10)

In [ ]:
# ============================================================
# CÉLULA 18 — Distribuição geográfica dos clientes VIP
# ============================================================
# value_counts() para entender de quais estados vêm os clientes
# mais fiéis — útil para o Marketing direcionar verba de mídia.

vip_por_estado = lista_vip['estado'].value_counts()
pct_vip = (vip_por_estado / len(lista_vip) * 100).round(1)

df_vip_geo = pd.DataFrame({
    'Clientes VIP': vip_por_estado,
    '% do total VIP': pct_vip
})

print('📍 Distribuição de Clientes VIP por Estado (Top 10):')
print(df_vip_geo.head(10).to_string())
print()
print('💡 Insight: o eixo SP–MG–RJ concentra a maioria dos clientes fiéis.')
print('   O Marketing pode priorizar mídia paga nesses estados.')

---
## ✅ Resumo — Casos Resolvidos

### 📣 Resposta para a Diretora de Marketing

> **A taxa de retenção NÃO é zero.** O dashboard usava `customer_id` (chave de pedido).  
> Com `customer_unique_id` identificamos **2.997 clientes recorrentes** prontos para o cupom da Black Friday. 🎉

### 🚚 Resposta para o Diretor de Logística

> **Dois problemas corrigidos:**  
> (1) Datas tipadas como `object` → corrigido com `pd.to_datetime()`  
> (2) Nulos em pedidos não entregues → removidos com `dropna(subset=[...])`  
> Resultado: tempo médio de entrega real = **12,1 dias** (mediana: 10 dias).

### 🔗 Lista VIP entregue

> Cruzamos `customers → orders → payments` com LEFT JOINs e pré-agregação segura.  
> **2.997 clientes elegíveis**, gasto médio R$ 314,99, top cliente gastou R$ 9.553.

---

## 🛠️ Ferramentas Utilizadas

| Função | Para que serviu |
|---|---|
| `df['col'].nunique()` | Contar valores distintos (revelar o paradoxo) |
| `df['col'].value_counts()` | Frequência de cada valor |
| `df.isnull().sum()` | Mapear valores nulos por coluna |
| `pd.to_datetime()` | Converter texto → data |
| `.dt.days` | Extrair dias de um timedelta |
| `df.dropna(subset=[...])` | Remover linhas com nulos em colunas específicas |
| `df.fillna()` | Preencher nulos com um valor |
| `df.groupby().agg()` | Agregar dados por grupo |
| `pd.merge()` | Cruzar tabelas (LEFT JOIN) |

---

## 🏋️ Exercícios de Fixação

**1.** Calcule a porcentagem de clientes recorrentes sobre o total de clientes únicos.

**2.** Adicione à `lista_vip` uma coluna `ticket_medio` = `total_gasto / total_pedidos`. Quem tem o maior ticket médio?

**3.** *(Desafio)* Calcule o tempo médio de entrega agrupado por `customer_state`. Qual estado tem o pior lead time?

**4.** *(Desafio)* Compare a receita total calculada com `payments_por_pedido` versus o merge errado da Célula 14. Qual é a diferença em R$?

---

## 🔮 Próxima Sessão — Métricas Operacionais

Na **Sessão 3** vamos aprofundar a análise logística:
- Comparar **promessa vs. realidade** (data estimada vs. data real)
- Criar um **flag de pedidos atrasados**
- Correlacionar atraso com **nota de avaliação** com Boxplots (Seaborn)